<a href="https://colab.research.google.com/github/hamza-24-ai/Pytorch_with_practical_deepLearning/blob/main/04_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Import data

In [5]:
df = pd.read_csv('/content/Colab_data/fashion-mnist_train.csv')
print(f"About data : {df.shape}")
print("\n Show the first five rows of data ")
print(df.head())

About data : (60000, 785)

 Show the first five rows of data 
   label  pixel1  pixel2  pixel3  pixel4  pixel5  pixel6  pixel7  pixel8  \
0      2       0       0       0       0       0       0       0       0   
1      9       0       0       0       0       0       0       0       0   
2      6       0       0       0       0       0       0       0       5   
3      0       0       0       0       1       2       0       0       0   
4      3       0       0       0       0       0       0       0       0   

   pixel9  ...  pixel775  pixel776  pixel777  pixel778  pixel779  pixel780  \
0       0  ...         0         0         0         0         0         0   
1       0  ...         0         0         0         0         0         0   
2       0  ...         0         0         0        30        43         0   
3       0  ...         3         0         0         0         0         1   
4       0  ...         0         0         0         0         0         0   

   pixel781 

In [6]:
# check the available Gpu

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
print(f"Device : {device} is running")

Device : cuda is running


# Preprocessing tha data

In [7]:
print("Find any attribute have null column")
print(df.isnull().sum())

Find any attribute have null column
label       0
pixel1      0
pixel2      0
pixel3      0
pixel4      0
           ..
pixel780    0
pixel781    0
pixel782    0
pixel783    0
pixel784    0
Length: 785, dtype: int64


In [8]:
# Split the data
X = df.drop(columns = ['label'])
Y = df['label']



In [9]:
# Train test splitting

X_train,X_test,y_train,y_test = train_test_split(X,Y, test_size=0.2, random_state=42)
print("Data is splitted successfully")

Data is splitted successfully


In [10]:
# Standardized the data manually

X_train = X_train/255.0
X_test = X_test/255.0

print("Well data is scaled successfully")

Well data is scaled successfully


# DataSet

In [11]:
# Create Custom Dataset

class DataSet(Dataset):

  def __init__(self,features,labels):
    self.features = torch.tensor(features.values, dtype=torch.float32)
    self.labels = torch.tensor(labels.values,dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]


In [12]:
# Created Train DataSet Object
tarindataset = DataSet(X_train,y_train)
len(tarindataset)

48000

In [13]:
# Created Test DataSey

testdataset = DataSet(X_test,y_test)
len(testdataset)

12000

In [14]:
# Create Train and Test Loader
train_loader = DataLoader(tarindataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(testdataset, batch_size=32, shuffle=False, pin_memory=True)

print("Data Loader is done Successful")
print(f"Type Train Loader : {type(train_loader.dataset)}")
print(f"Type Test Loader : {type(test_loader.dataset)}")

Data Loader is done Successful
Type Train Loader : <class '__main__.DataSet'>
Type Test Loader : <class '__main__.DataSet'>


In [15]:
# Define NN Class

class MyNN(nn.Module):
  def __init__(self, num_features):

    super().__init__()

    self.module = nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10)
    )

  def forward(self,x):
    return self.module(x)


In [16]:
# Set Learning and Epochs
learning_rate=0.1
epochs = 100

In [17]:
# Initiate the model

model = MyNN(X_train.shape[1])
model = model.to(device)

# Loss Initiate
criterian = nn.CrossEntropyLoss()

# Initiate Optimizer

optimizer = optim.SGD(model.parameters(), lr=learning_rate)


In [18]:
# Training Loop

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features,batch_labels in train_loader:

    # Move data to GPU
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)

    # Forward Pass
    outputs = model(batch_features)

    # Loss
    loss = criterian(outputs,batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # Update Data
    optimizer.step()

    total_epoch_loss += loss.item()

  avg_loss = total_epoch_loss/len(train_loader)

  print(f" Epochs : {epoch+1}, Loss : {avg_loss}")

 Epochs : 1, Loss : 0.6343203376034896
 Epochs : 2, Loss : 0.42896964510778585
 Epochs : 3, Loss : 0.38609426518778006
 Epochs : 4, Loss : 0.3577473446056247
 Epochs : 5, Loss : 0.33706610773007073
 Epochs : 6, Loss : 0.32049589602152506
 Epochs : 7, Loss : 0.30650197458515566
 Epochs : 8, Loss : 0.29592895340919495
 Epochs : 9, Loss : 0.2837557953049739
 Epochs : 10, Loss : 0.27399841212232906
 Epochs : 11, Loss : 0.2636327679802974
 Epochs : 12, Loss : 0.25751938185095785
 Epochs : 13, Loss : 0.24993605478480457
 Epochs : 14, Loss : 0.24142360664655765
 Epochs : 15, Loss : 0.23522054675407708
 Epochs : 16, Loss : 0.23085723438858985
 Epochs : 17, Loss : 0.22546787700802087
 Epochs : 18, Loss : 0.2172601928760608
 Epochs : 19, Loss : 0.21422301924663287
 Epochs : 20, Loss : 0.2085672876300911
 Epochs : 21, Loss : 0.20350302611663937
 Epochs : 22, Loss : 0.19651326823607088
 Epochs : 23, Loss : 0.19541106336315472
 Epochs : 24, Loss : 0.190376487341399
 Epochs : 25, Loss : 0.1844048453

In [19]:
# write Evaluation Code

model.eval()

MyNN(
  (module): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [20]:
# Find accuracy
total = 0
correct = 0

with torch.no_grad():

  for batch_features,batch_labels in test_loader:
     # Move data to GPU
    batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs,1)

    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f"Accuracy Score : {correct/total}")

Accuracy Score : 0.88625
